# Semantic Versioning and Release Automation

**Semantic versioning** (SemVer) is the industry standard for communicating what changed in a software release. The format is **MAJOR.MINOR.PATCH** (e.g., `2.1.3`).

- **MAJOR**: Breaking change -- users must update their code to continue working
- **MINOR**: New feature, backward compatible -- users can upgrade safely, nothing breaks
- **PATCH**: Bug fix, backward compatible -- safe upgrade, no new features

This notebook covers:
1. Conventional Commits -- the standard for commit messages that enable automated versioning
2. Automated version bumping
3. Changelog generation
4. GitHub Actions release workflow
5. Python package versioning
6. ML model versioning

In [1]:
import re
import json
import subprocess
from datetime import datetime
from typing import List, Optional, Dict, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum

## Section 1: Conventional Commits

**Conventional Commits** is a specification that adds human-readable and machine-parseable meaning to commit messages. It was designed specifically to enable automated versioning and changelog generation.

### Format

```
type(scope): description

[optional body]

[optional footer(s)]
```

### Commit Types and Version Impact

| Type | Description | Version Bump |
|------|-------------|-------------|
| `feat` | New feature | MINOR |
| `fix` | Bug fix | PATCH |
| `BREAKING CHANGE` | Breaking API change (in footer) | MAJOR |
| `docs` | Documentation only | None |
| `chore` | Maintenance, dependencies | None |
| `refactor` | Refactoring (no new feature or fix) | None |
| `test` | Adding or updating tests | None |
| `perf` | Performance improvement | PATCH |
| `ci` | CI/CD configuration | None |

### Examples

```
feat(api): add batch prediction endpoint

fix(model): correct normalization bug in preprocessing

feat(auth)!: require API key for all endpoints

BREAKING CHANGE: unauthenticated access no longer permitted

docs(readme): update quickstart guide

chore(deps): upgrade transformers to 4.40.0
```

In [2]:
class VersionBump(str, Enum):
    MAJOR = "major"
    MINOR = "minor"
    PATCH = "patch"
    NONE = "none"


@dataclass
class ParsedCommit:
    raw: str
    type: Optional[str]
    scope: Optional[str]
    description: str
    body: str
    is_breaking: bool
    version_bump: VersionBump
    sha: str = ""


# Conventional commit regex
# Matches: type(scope)!: description OR type!: description OR type: description
CONVENTIONAL_COMMIT_RE = re.compile(
    r'^(?P<type>feat|fix|docs|chore|refactor|test|perf|ci|build|style|revert)'
    r'(?:\((?P<scope>[^)]+)\))?'
    r'(?P<breaking>!)?'
    r':\s*(?P<description>.+)$',
    re.IGNORECASE
)

TYPE_TO_BUMP = {
    "feat": VersionBump.MINOR,
    "fix": VersionBump.PATCH,
    "perf": VersionBump.PATCH,
    "revert": VersionBump.PATCH,
    "docs": VersionBump.NONE,
    "chore": VersionBump.NONE,
    "refactor": VersionBump.NONE,
    "test": VersionBump.NONE,
    "ci": VersionBump.NONE,
    "build": VersionBump.NONE,
    "style": VersionBump.NONE
}


def parse_commit_message(raw_message: str, sha: str = "") -> ParsedCommit:
    """Parse a conventional commit message into its components."""
    lines = raw_message.strip().split("\n")
    subject = lines[0]
    body = "\n".join(lines[1:]).strip() if len(lines) > 1 else ""

    # Check for BREAKING CHANGE in footer
    has_breaking_in_footer = bool(re.search(r'^BREAKING[ -]CHANGE:', body, re.MULTILINE))

    match = CONVENTIONAL_COMMIT_RE.match(subject)
    if not match:
        return ParsedCommit(
            raw=raw_message,
            type=None,
            scope=None,
            description=subject,
            body=body,
            is_breaking=False,
            version_bump=VersionBump.NONE,
            sha=sha
        )

    commit_type = match.group("type").lower()
    scope = match.group("scope")
    description = match.group("description")
    is_breaking = bool(match.group("breaking")) or has_breaking_in_footer

    if is_breaking:
        version_bump = VersionBump.MAJOR
    else:
        version_bump = TYPE_TO_BUMP.get(commit_type, VersionBump.NONE)

    return ParsedCommit(
        raw=raw_message,
        type=commit_type,
        scope=scope,
        description=description,
        body=body,
        is_breaking=is_breaking,
        version_bump=version_bump,
        sha=sha
    )


# Test with various commit messages
test_commits = [
    "feat(api): add batch prediction endpoint",
    "fix(model): correct normalization bug in preprocessing",
    "feat(auth)!: require API key for all endpoints",
    "docs(readme): update quickstart guide",
    "chore(deps): upgrade transformers to 4.40.0",
    "refactor(data): simplify DataLoader class",
    "perf(inference): vectorize token embedding lookup",
    "not a conventional commit at all"
]

print("Parsed Conventional Commits:")
print("-" * 70)
for msg in test_commits:
    parsed = parse_commit_message(msg)
    scope_str = f"({parsed.scope})" if parsed.scope else ""
    breaking_str = " [BREAKING]" if parsed.is_breaking else ""
    print(f"  {parsed.type or '?'}{scope_str}{breaking_str} -> {parsed.version_bump.value.upper()} | {parsed.description}")

Parsed Conventional Commits:
----------------------------------------------------------------------
  feat(api) -> MINOR | add batch prediction endpoint
  fix(model) -> PATCH | correct normalization bug in preprocessing
  feat(auth) [BREAKING] -> MAJOR | require API key for all endpoints
  docs(readme) -> NONE | update quickstart guide
  chore(deps) -> NONE | upgrade transformers to 4.40.0
  refactor(data) -> NONE | simplify DataLoader class
  perf(inference) -> PATCH | vectorize token embedding lookup
  ? -> NONE | not a conventional commit at all


## Section 2: Automated Version Bumping

Given a set of commits since the last tag, we can deterministically compute the next version:

- Any `BREAKING CHANGE` -> increment MAJOR, reset MINOR and PATCH to 0
- Any `feat` (no breaking) -> increment MINOR, reset PATCH to 0
- Only `fix`, `perf`, `revert` -> increment PATCH
- Only `docs`, `chore`, `refactor`, `test`, `ci` -> no version change

In [3]:
@dataclass
class Version:
    major: int
    minor: int
    patch: int
    prerelease: str = ""

    @classmethod
    def parse(cls, version_str: str) -> "Version":
        """Parse a version string like '1.2.3' or 'v1.2.3-alpha.1'."""
        s = version_str.lstrip("v")
        # Split prerelease
        prerelease = ""
        if "-" in s:
            s, prerelease = s.split("-", 1)
        parts = s.split(".")
        return cls(
            major=int(parts[0]),
            minor=int(parts[1]) if len(parts) > 1 else 0,
            patch=int(parts[2]) if len(parts) > 2 else 0,
            prerelease=prerelease
        )

    def __str__(self) -> str:
        base = f"{self.major}.{self.minor}.{self.patch}"
        return f"{base}-{self.prerelease}" if self.prerelease else base


def determine_next_version(
    commits: List[ParsedCommit],
    current_version: str
) -> Tuple[Version, VersionBump, List[str]]:
    """Analyze commits since last tag and determine the next version.

    Returns:
        (next_version, bump_type, reasons)
    """
    current = Version.parse(current_version)
    reasons = []

    has_breaking = any(c.is_breaking for c in commits)
    has_feat = any(c.type == "feat" and not c.is_breaking for c in commits)
    has_fix = any(c.version_bump == VersionBump.PATCH for c in commits)

    if has_breaking:
        bump_type = VersionBump.MAJOR
        breaking_commits = [c.description for c in commits if c.is_breaking]
        reasons.append(f"BREAKING CHANGE detected: {', '.join(breaking_commits)}")
        next_version = Version(major=current.major + 1, minor=0, patch=0)
    elif has_feat:
        bump_type = VersionBump.MINOR
        feat_commits = [c.description for c in commits if c.type == "feat"]
        reasons.append(f"New features: {', '.join(feat_commits[:3])}")
        next_version = Version(major=current.major, minor=current.minor + 1, patch=0)
    elif has_fix:
        bump_type = VersionBump.PATCH
        fix_commits = [c.description for c in commits if c.version_bump == VersionBump.PATCH]
        reasons.append(f"Bug fixes: {', '.join(fix_commits[:3])}")
        next_version = Version(major=current.major, minor=current.minor, patch=current.patch + 1)
    else:
        bump_type = VersionBump.NONE
        reasons.append("No releasable changes (only docs, chores, refactoring)")
        next_version = current

    return next_version, bump_type, reasons


# Simulate commit history since last release (v1.2.3)
commit_history = [
    "feat(api): add batch prediction endpoint",
    "fix(model): correct normalization bug in preprocessing",
    "docs(api): update endpoint documentation",
    "chore(deps): upgrade numpy to 1.26",
    "refactor(data): clean up DataLoader",
    "test(api): add integration tests for batch endpoint"
]
commits = [parse_commit_message(msg) for msg in commit_history]
next_ver, bump, reasons = determine_next_version(commits, "1.2.3")

print("Version Analysis")
print("=" * 50)
print(f"Current version: 1.2.3")
print(f"Commits analyzed: {len(commits)}")
print(f"Bump type: {bump.value.upper()}")
print(f"Next version: {next_ver}")
print(f"Reason: {reasons[0]}")
print()

# Now with a BREAKING CHANGE
breaking_commits_raw = [
    "feat(auth)!: require API key for all endpoints",
    "fix(logging): handle empty log entries"
]
breaking_commits = [parse_commit_message(msg) for msg in breaking_commits_raw]
next_ver2, bump2, reasons2 = determine_next_version(breaking_commits, "1.2.3")
print(f"With BREAKING CHANGE: 1.2.3 -> {next_ver2} ({bump2.value.upper()})")
print(f"Reason: {reasons2[0]}")

Version Analysis
Current version: 1.2.3
Commits analyzed: 6
Bump type: MINOR
Next version: 1.3.0
Reason: New features: add batch prediction endpoint

With BREAKING CHANGE: 1.2.3 -> 2.0.0 (MAJOR)
Reason: BREAKING CHANGE detected: require API key for all endpoints


## Section 3: Changelog Generation

The **Keep a Changelog** format (keepachangelog.com) is the standard for human-readable changelogs. Conventional commits map cleanly to its sections.

In [4]:
def generate_changelog(
    commits: List[ParsedCommit],
    version: str,
    date: Optional[str] = None,
    repo_url: str = "https://github.com/example/myproject"
) -> str:
    """Generate a CHANGELOG.md entry in Keep a Changelog format."""
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")

    groups: Dict[str, List[ParsedCommit]] = {
        "Breaking Changes": [],
        "Features": [],
        "Bug Fixes": [],
        "Performance": [],
        "Other": []
    }

    for commit in commits:
        if commit.is_breaking:
            groups["Breaking Changes"].append(commit)
        elif commit.type == "feat":
            groups["Features"].append(commit)
        elif commit.type in ("fix", "revert"):
            groups["Bug Fixes"].append(commit)
        elif commit.type == "perf":
            groups["Performance"].append(commit)
        elif commit.type in ("docs", "chore", "refactor", "test", "ci", "build", "style"):
            groups["Other"].append(commit)

    lines = [
        f"## [{version}] - {date}",
        ""
    ]

    for section, section_commits in groups.items():
        if not section_commits:
            continue
        lines.append(f"### {section}")
        lines.append("")
        for c in section_commits:
            scope_prefix = f"**{c.scope}**: " if c.scope else ""
            sha_suffix = f" ([{c.sha[:7]}]({repo_url}/commit/{c.sha}))" if c.sha else ""
            lines.append(f"- {scope_prefix}{c.description}{sha_suffix}")
        lines.append("")

    lines.append(f"[{version}]: {repo_url}/compare/v{version}...HEAD")

    return "\n".join(lines)


# Full release with all commit types
full_release_commits_raw = [
    ("feat(api): add batch prediction endpoint", "abc1234"),
    ("feat(model): add ONNX export support", "def5678"),
    ("fix(model): correct normalization bug in preprocessing", "ghi9012"),
    ("fix(api): handle empty input lists gracefully", "jkl3456"),
    ("perf(inference): vectorize token embedding lookup", "mno7890"),
    ("docs(api): update endpoint documentation", "pqr1234"),
    ("chore(deps): upgrade transformers to 4.40.0", "stu5678")
]

full_commits = [parse_commit_message(msg, sha) for msg, sha in full_release_commits_raw]
changelog = generate_changelog(full_commits, version="1.3.0", date="2024-11-15")

print(changelog)

## [1.3.0] - 2024-11-15

### Features

- **api**: add batch prediction endpoint ([abc1234](https://github.com/example/myproject/commit/abc1234))
- **model**: add ONNX export support ([def5678](https://github.com/example/myproject/commit/def5678))

### Bug Fixes

- **model**: correct normalization bug in preprocessing ([ghi9012](https://github.com/example/myproject/commit/ghi9012))
- **api**: handle empty input lists gracefully ([jkl3456](https://github.com/example/myproject/commit/jkl3456))

### Performance

- **inference**: vectorize token embedding lookup ([mno7890](https://github.com/example/myproject/commit/mno7890))

### Other

- **api**: update endpoint documentation ([pqr1234](https://github.com/example/myproject/commit/pqr1234))
- **deps**: upgrade transformers to 4.40.0 ([stu5678](https://github.com/example/myproject/commit/stu5678))

[1.3.0]: https://github.com/example/myproject/compare/v1.3.0...HEAD


## Section 4: GitHub Actions Release Workflow

The following GitHub Actions workflow automates the full release pipeline: test, version bump, changelog, tag, and publish.

In [5]:
release_workflow_yaml = '''
name: Release

on:
  push:
    branches:
      - main

permissions:
  contents: write
  pull-requests: write

jobs:
  release:
    name: Release
    runs-on: ubuntu-latest

    steps:
      # Step 1: Checkout full history so we can analyze all commits
      - name: Checkout
        uses: actions/checkout@v4
        with:
          fetch-depth: 0  # Full history required for conventional-changelog

      # Step 2: Set up Python and install dependencies
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest

      # Step 3: Run tests -- release only happens if tests pass
      - name: Run tests
        run: pytest tests/ -v --tb=short

      # Step 4: Analyze commits and determine next version
      # Uses semantic-release to parse conventional commits
      - name: semantic-release
        uses: python-semantic-release/python-semantic-release@master
        with:
          github_token: ${{ secrets.GITHUB_TOKEN }}
          # This action will:
          # 1. Parse commits since last tag
          # 2. Determine version bump (major/minor/patch)
          # 3. Update pyproject.toml version
          # 4. Generate CHANGELOG.md
          # 5. Create git tag (e.g. v1.3.0)
          # 6. Create GitHub Release with notes
          # 7. Optionally publish to PyPI

      # Step 5: Publish to PyPI (optional)
      - name: Publish to PyPI
        if: steps.semantic-release.outputs.released == 'true'
        uses: pypa/gh-action-pypi-publish@release/v1
        with:
          password: ${{ secrets.PYPI_API_TOKEN }}
'''

print("GitHub Actions Release Workflow (.github/workflows/release.yml):")
print("=" * 70)
print(release_workflow_yaml)
print("\nWorkflow breakdown:")
steps = [
    ("actions/checkout", "Full git history needed to find last tag and all commits since"),
    ("actions/setup-python", "Install Python environment for running tests"),
    ("pytest", "Tests must pass before any release -- gate to production"),
    ("python-semantic-release", "Parse conventional commits, bump version, update pyproject.toml, generate CHANGELOG.md, create tag and GitHub Release"),
    ("pypi-publish", "Publish the new version to PyPI (only if a release was made)")
]
for step, description in steps:
    print(f"  {step}: {description}")

GitHub Actions Release Workflow (.github/workflows/release.yml):

name: Release

on:
  push:
    branches:
      - main

permissions:
  contents: write
  pull-requests: write

jobs:
  release:
    name: Release
    runs-on: ubuntu-latest

    steps:
      # Step 1: Checkout full history so we can analyze all commits
      - name: Checkout
        uses: actions/checkout@v4
        with:
          fetch-depth: 0  # Full history required for conventional-changelog

      # Step 2: Set up Python and install dependencies
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest

      # Step 3: Run tests -- release only happens if tests pass
      - name: Run tests
        run: pytest tests/ -v --tb=short

      # Step 4: Analyze commits and determine next version
      # Uses semantic-release to parse conventional c

## Section 5: Python Package Versioning

There are three approaches to versioning Python packages:

1. **Static version in pyproject.toml** -- Simple, manual, error-prone
2. **`__version__` in package** -- Traditional approach, must sync manually
3. **Dynamic versioning with setuptools-scm** -- Derives version from git tags automatically

In [6]:
# Approach 1: Static version in pyproject.toml
pyproject_static = '''
[project]
name = "myml-package"
version = "1.3.0"  # Update manually with each release
requires-python = ">=3.9"
dependencies = [
    "torch>=2.0",
    "transformers>=4.40",
]

[tool.semantic_release]
version_variable = "myml/__init__.py:__version__"
version_toml = "pyproject.toml:project.version"
branch = "main"
changelog_file = "CHANGELOG.md"
build_command = "pip install build && python -m build"
'''

print("Approach 1: Static version in pyproject.toml")
print(pyproject_static)

# The __version__ variable
package_init = '''
# myml/__init__.py
__version__ = "1.3.0"  # Keep in sync with pyproject.toml

from myml.model import Model
from myml.pipeline import Pipeline

__all__ = ["Model", "Pipeline", "__version__"]
'''

print("\npackage __init__.py:")
print(package_init)

Approach 1: Static version in pyproject.toml

[project]
name = "myml-package"
version = "1.3.0"  # Update manually with each release
requires-python = ">=3.9"
dependencies = [
    "torch>=2.0",
    "transformers>=4.40",
]

[tool.semantic_release]
version_variable = "myml/__init__.py:__version__"
version_toml = "pyproject.toml:project.version"
branch = "main"
changelog_file = "CHANGELOG.md"
build_command = "pip install build && python -m build"


package __init__.py:

# myml/__init__.py
__version__ = "1.3.0"  # Keep in sync with pyproject.toml

from myml.model import Model
from myml.pipeline import Pipeline

__all__ = ["Model", "Pipeline", "__version__"]



In [7]:
# Approach 3: Dynamic versioning with setuptools-scm
pyproject_dynamic = '''
[project]
name = "myml-package"
dynamic = ["version"]  # Version comes from git tags
requires-python = ">=3.9"

[tool.setuptools_scm]
# Version is derived from git tags automatically.
# No manual version bumping needed.
# git tag v1.3.0 && git push origin v1.3.0
# -> package version becomes 1.3.0
# Between tags: 1.3.0.dev5+g3a5b7c9 (dev build)
write_to = "myml/_version.py"  # Optional: write version to file
fallback_version = "0.0.0"

[build-system]
requires = ["setuptools>=64", "setuptools-scm>=8"]
build-backend = "setuptools.backends.legacy:build"
'''

print("Approach 3: Dynamic versioning with setuptools-scm (recommended)")
print(pyproject_dynamic)

# Check current version from git tags (if in a git repo)
print("Current git tags (if in a git repo):")
result = subprocess.run(
    ["git", "tag", "--sort=-version:refname"],
    capture_output=True, text=True
)
if result.returncode == 0 and result.stdout.strip():
    tags = result.stdout.strip().split("\n")[:5]
    for tag in tags:
        print(f"  {tag}")
else:
    print("  (not in a git repo, or no tags found)")

Approach 3: Dynamic versioning with setuptools-scm (recommended)

[project]
name = "myml-package"
dynamic = ["version"]  # Version comes from git tags
requires-python = ">=3.9"

[tool.setuptools_scm]
# Version is derived from git tags automatically.
# No manual version bumping needed.
# git tag v1.3.0 && git push origin v1.3.0
# -> package version becomes 1.3.0
# Between tags: 1.3.0.dev5+g3a5b7c9 (dev build)
write_to = "myml/_version.py"  # Optional: write version to file
fallback_version = "0.0.0"

[build-system]
requires = ["setuptools>=64", "setuptools-scm>=8"]
build-backend = "setuptools.backends.legacy:build"

Current git tags (if in a git repo):
  (not in a git repo, or no tags found)


## Section 6: ML Model Versioning vs Software Versioning

Software versioning tracks code changes. ML model versioning must track three things that together determine model behavior:

| Component | Why it matters |
|-----------|----------------|
| **Code version** | Architecture, preprocessing, postprocessing |
| **Data version** | Training data distribution; same code + new data = different model |
| **Hyperparameters** | Learning rate, batch size, epochs, regularization |

Two models with the same code version but different data or hyperparameters will behave differently. All three must be versioned together.

In [8]:
@dataclass
class ModelVersion:
    """Complete version specification for an ML model."""
    model_name: str
    code_version: str      # From git tag / pyproject.toml
    data_version: str      # Data pipeline version or dataset hash
    hyperparameters: Dict[str, Any]
    training_date: str = field(default_factory=lambda: datetime.now().isoformat()[:10])
    mlflow_run_id: str = ""  # MLflow tracking URI
    docker_image_tag: str = ""
    metrics: Dict[str, float] = field(default_factory=dict)

    @property
    def full_version(self) -> str:
        """Canonical model version string."""
        return f"{self.code_version}_data-{self.data_version}"

    def to_docker_tag(self, registry: str, repo: str) -> str:
        """Generate a Docker image tag for this model version."""
        # Include model version in Docker tag for full traceability
        safe_version = self.full_version.replace("+", "-").replace(".", "-")
        return f"{registry}/{repo}:{self.model_name}-{safe_version}"

    def to_mlflow_tags(self) -> Dict[str, str]:
        """Generate MLflow tags for model registry."""
        return {
            "model.code_version": self.code_version,
            "model.data_version": self.data_version,
            "model.training_date": self.training_date,
            "model.full_version": self.full_version
        }


# Example: two model versions with same code but different data
v1 = ModelVersion(
    model_name="sentiment-classifier",
    code_version="1.3.0",
    data_version="2024-09",  # September dataset
    hyperparameters={"lr": 2e-5, "epochs": 3, "batch_size": 16},
    metrics={"accuracy": 0.921, "f1": 0.918},
    docker_image_tag=""
)
v1.docker_image_tag = v1.to_docker_tag("registry.company.com", "ml-models")

v2 = ModelVersion(
    model_name="sentiment-classifier",
    code_version="1.3.0",  # Same code
    data_version="2024-11",  # November dataset -- more data
    hyperparameters={"lr": 2e-5, "epochs": 3, "batch_size": 16},
    metrics={"accuracy": 0.934, "f1": 0.931},
    docker_image_tag=""
)
v2.docker_image_tag = v2.to_docker_tag("registry.company.com", "ml-models")

print("ML Model Version Comparison")
print("=" * 60)
for v in [v1, v2]:
    print(f"\nModel: {v.model_name}")
    print(f"  Full version: {v.full_version}")
    print(f"  Code version: {v.code_version}")
    print(f"  Data version: {v.data_version}")
    print(f"  Accuracy: {v.metrics.get('accuracy', 'N/A')}")
    print(f"  Docker tag: {v.docker_image_tag}")
    print(f"  MLflow tags: {v.to_mlflow_tags()}")

print("\nKey insight: same code (1.3.0) but different data produces different models.")
print("Both code AND data version must be tracked for reproducibility.")

ML Model Version Comparison

Model: sentiment-classifier
  Full version: 1.3.0_data-2024-09
  Code version: 1.3.0
  Data version: 2024-09
  Accuracy: 0.921
  Docker tag: registry.company.com/ml-models:sentiment-classifier-1-3-0_data-2024-09
  MLflow tags: {'model.code_version': '1.3.0', 'model.data_version': '2024-09', 'model.training_date': '2026-06-29', 'model.full_version': '1.3.0_data-2024-09'}

Model: sentiment-classifier
  Full version: 1.3.0_data-2024-11
  Code version: 1.3.0
  Data version: 2024-11
  Accuracy: 0.934
  Docker tag: registry.company.com/ml-models:sentiment-classifier-1-3-0_data-2024-11
  MLflow tags: {'model.code_version': '1.3.0', 'model.data_version': '2024-11', 'model.training_date': '2026-06-29', 'model.full_version': '1.3.0_data-2024-11'}

Key insight: same code (1.3.0) but different data produces different models.
Both code AND data version must be tracked for reproducibility.


In [9]:
# MLflow model registry pattern
mlflow_registry_example = '''
# Registering a model with MLflow
import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    # Log hyperparameters
    mlflow.log_param("lr", 2e-5)
    mlflow.log_param("epochs", 3)
    mlflow.log_param("data_version", "2024-11")

    # Train model...
    # model = train()

    # Log metrics
    mlflow.log_metric("accuracy", 0.934)
    mlflow.log_metric("f1", 0.931)

    # Log model with version tags
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sentiment_classifier",
        registered_model_name="SentimentClassifier"
    )

# Transition version to production
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name="SentimentClassifier",
    version=2,  # The version number assigned by MLflow
    stage="Production"
)

# Load model from registry
model = mlflow.sklearn.load_model("models:/SentimentClassifier/Production")
'''

print("MLflow Model Registry Pattern:")
print(mlflow_registry_example)

MLflow Model Registry Pattern:

# Registering a model with MLflow
import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    # Log hyperparameters
    mlflow.log_param("lr", 2e-5)
    mlflow.log_param("epochs", 3)
    mlflow.log_param("data_version", "2024-11")

    # Train model...
    # model = train()

    # Log metrics
    mlflow.log_metric("accuracy", 0.934)
    mlflow.log_metric("f1", 0.931)

    # Log model with version tags
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sentiment_classifier",
        registered_model_name="SentimentClassifier"
    )

# Transition version to production
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name="SentimentClassifier",
    version=2,  # The version number assigned by MLflow
    stage="Production"
)

# Load model from registry
model = mlflow.sklearn.load_model("models:/SentimentClassifier/Production")



## Key Takeaways

1. **SemVer communicates change impact**: MAJOR.MINOR.PATCH tells users exactly how much to worry about upgrading. Use it consistently so your users trust version numbers.

2. **Conventional Commits enable full automation**: With structured commit messages, a machine can determine the next version, write the changelog, create the tag, and publish a release -- no human judgment needed.

3. **The automation chain**: commit message -> version bump type -> next version -> CHANGELOG.md -> git tag -> GitHub Release -> PyPI publish. This entire chain can be triggered automatically on push to main.

4. **setuptools-scm removes manual version syncing**: When the version lives in a git tag, there is no risk of pyproject.toml and `__version__` getting out of sync.

5. **ML models need three-part versioning**: code version + data version + hyperparameters together specify a model. Tracking only code version leads to unreproducible experiments.

6. **Docker tags should include model version**: `registry/repo:modelname-code_version-data_version` creates a direct link between the running container and the exact model that is inside it.

7. **MLflow is the standard for ML experiment tracking**: It handles model versioning, metric logging, and stage transitions (staging/production/archived) for model serving.